# 09 - Frontend Validation

Check Streamlit page syntax and artifact readiness before launching the dashboard.

## 1. Setup

In [1]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
from IPython.display import display, Image, HTML


def resolve_project_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent, Path('..').resolve()]
    for candidate in candidates:
        if (candidate / 'src').exists() and (candidate / 'configs' / 'config.yaml').exists():
            return candidate
    raise FileNotFoundError('Could not resolve project root. Run this notebook from the repo or notebooks folder.')

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import load_config
from src.pipeline_reporting import configure_pretty_logging, output_dir, save_dataset_inventory, save_dataframe, save_table_image, save_bar_chart

config = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')
paths = config['paths']
processed_dir = PROJECT_ROOT / paths.get('processed_data_dir', 'data/processed')
raw_dir = PROJECT_ROOT / paths.get('raw_data_dir', 'data/raw')
outputs_dir = PROJECT_ROOT / 'outputs'
notebook_out = output_dir('09_frontend_validation', root=outputs_dir / 'notebook_reports')
log_file = configure_pretty_logging(PROJECT_ROOT / 'logs', '09_frontend_validation')
print('Project root:', PROJECT_ROOT)
print('Notebook output dir:', notebook_out)
print('Notebook log file:', log_file)

Project root: C:\Code\flight-disruption-prediction
Notebook output dir: C:\Code\flight-disruption-prediction\outputs\notebook_reports\09_frontend_validation
Notebook log file: C:\Code\flight-disruption-prediction\logs\09_frontend_validation_20260420_173506.log


## 2. Syntax Smoke Test

In [2]:
import ast, py_compile
page_files = sorted((PROJECT_ROOT / 'pages').glob('*.py'))
app_path = PROJECT_ROOT / 'app.py'
rows = []
for path in [app_path] + page_files:
    row = {'file': str(path.relative_to(PROJECT_ROOT)), 'syntax_ok': False, 'ast_ok': False, 'error': ''}
    try:
        py_compile.compile(str(path), doraise=True)
        row['syntax_ok'] = True
        ast.parse(path.read_text(encoding='utf-8'))
        row['ast_ok'] = True
    except Exception as exc:
        row['error'] = repr(exc)
    rows.append(row)
syntax_df = pd.DataFrame(rows)
save_dataframe(syntax_df, 'frontend_syntax_check', notebook_out)
display(syntax_df)

,file,syntax_ok,ast_ok,error
0,app.py,True,True,
1,pages\1_Pipeline_Overview.py,True,True,
2,pages\2_Data_Explorer.py,True,True,
3,pages\3_Trajectory_Map.py,True,True,
4,pages\4_Feature_Analysis.py,True,True,
5,pages\5_Model_Performance.py,True,True,
6,pages\6_Predictions_Explorer.py,True,True,
7,pages\7_Data_Quality.py,True,True,


## 3. Artifact Readiness

In [3]:
checks = {
    'ml_dataset': processed_dir / paths.get('ml_dataset_file', 'ml_dataset.parquet'),
    'trajectory_features': processed_dir / paths.get('features_file', 'trajectory_features.parquet'),
    'model_comparison': PROJECT_ROOT / 'logs' / 'model_comparison.json',
    'roc_curves': PROJECT_ROOT / 'outputs' / 'roc_curves.png',
    'shap_summary': PROJECT_ROOT / 'outputs' / 'shap_summary.png',
    'scaler': PROJECT_ROOT / 'models' / 'scaler.pkl',
    'imputer': PROJECT_ROOT / 'models' / 'imputer.pkl',
    'label_encoder': PROJECT_ROOT / 'models' / 'label_encoder.pkl',
}
ready = pd.DataFrame([{'artifact': k, 'path': str(v.relative_to(PROJECT_ROOT)), 'exists': v.exists(), 'size_mb': round(v.stat().st_size/1024**2, 3) if v.exists() else 0} for k,v in checks.items()])
save_dataframe(ready, 'frontend_artifact_readiness', notebook_out)
display(ready)

,artifact,path,exists,size_mb
0,ml_dataset,data\processed\ml_dataset.parquet,True,10.365
1,trajectory_features,data\processed\trajectory_features.parquet,True,11.867
2,model_comparison,logs\model_comparison.json,True,0.002
3,roc_curves,outputs\roc_curves.png,True,0.115
4,shap_summary,outputs\shap_summary.png,True,0.218
5,scaler,models\scaler.pkl,True,0.002
6,imputer,models\imputer.pkl,True,0.002
7,label_encoder,models\label_encoder.pkl,True,0.000


## 4. Launch Command

In [4]:
print('streamlit run app.py')

streamlit run app.py
